Transformações para a camada Gold  
1-Selecionar apenas colunas especificas  
2-Criar uma nova coluna que faz a soma das lesoes  
3-Renomear as colunas para ficar intuitivo o nome  
4-Excluir dados que tenham na classificação ['Indeterminado', 'Sem Registro','Exterior']  
5-Inserir coluna com nome de atualização para usuario ver quandos os dados foram atualizados
6-Salvar na camada Gold particionada por UF > 'Estado'

In [0]:
%fs ls /Volumes/anac/databricks/anac_volume

In [0]:
df= spark.read.parquet("/Volumes/anac/databricks/anac_volume/02.Silver/")
display(df)

In [0]:
print(df.columns)

In [0]:
# 1-Secionar colunas
colunas = ['Aerodromo_de_Destino', 'Aerodromo_de_Origem', 'Classificacao_da_Ocorrência', 'Danos_a_Aeronave', 'Data_da_Ocorrencia','Municipio', 'UF', 'Regiao','Tipo_de_Aerodromo','Tipo_de_Ocorrencia', 'Lesoes_Desconhecidas_Passageiros','Lesoes_Desconhecidas_Terceiros', 'Lesoes_Desconhecidas_Tripulantes', 'Lesoes_Fatais_Passageiros', 'Lesoes_Fatais_Terceiros', 'Lesoes_Fatais_Tripulantes', 'Lesoes_Graves_Passageiros', 'Lesoes_Graves_Terceiros', 'Lesoes_Graves_Tripulantes', 'Lesoes_Leves_Passageiros', 'Lesoes_Leves_Terceiros', 'Lesoes_Leves_Tripulantes']

df= df.select(colunas)
display(df)

In [0]:
# 2-somar
colunas_somar = ['Lesoes_Desconhecidas_Passageiros','Lesoes_Desconhecidas_Terceiros', 'Lesoes_Desconhecidas_Tripulantes', 'Lesoes_Fatais_Passageiros', 'Lesoes_Fatais_Terceiros', 'Lesoes_Fatais_Tripulantes', 'Lesoes_Graves_Passageiros', 'Lesoes_Graves_Terceiros', 'Lesoes_Graves_Tripulantes', 'Lesoes_Leves_Passageiros', 'Lesoes_Leves_Terceiros', 'Lesoes_Leves_Tripulantes']

df = df.withColumn("Total_lesoes",sum(df[coluna] for coluna in colunas_somar))
display(df)


In [0]:
print(df.columns)

In [0]:
#3-renomear
df= df\
    .withColumnRenamed('Aerodromo_de_Destino', 'Destino')\
    .withColumnRenamed('Aerodromo_de_Origem', 'Origem')\
    .withColumnRenamed('Classificacao_da_Ocorrência', 'Classificacao')\
    .withColumnRenamed('Danos_a_Aeronave', 'Danos')\
    .withColumnRenamed('Data_da_Ocorrencia', 'Data')\
    .withColumnRenamed('UF','Estado')
display(df)


In [0]:
display(df.select("Estado").distinct())

In [0]:
# 4-excluir dados
cassificacoes_excluir = ['Indeterminado','Sem Registro','Exterior']
df = df.filter(~df.Estado.isin(cassificacoes_excluir))
display(df)

In [0]:
# 5-Inserir coluna data
from pyspark.sql.functions import current_timestamp,date_format,from_utc_timestamp
df = df.withColumn("Atualizacao",\
    date_format(from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"),"yyyy-MM-dd HH:mm:ss"))
display(df)

In [0]:
%fs ls /Volumes/anac/databricks/anac_volume

In [0]:
# 6-Salvar tudo na camada Gold particionada por UF > Estado
df.write\
    .mode("overwrite")\
    .format("parquet")\
    .partitionBy("Estado")\
    .save("/Volumes/anac/databricks/anac_volume/03.Gold/")